### **Setup (only for this file)**
These lines repeat what Parts 1 to 4 already do: the setup, the functions of Part 2, the candidates of 21.1, the final set of Part 3 (28.3) and the final model of Part 4, trained on 2015 to 2023 as in section 34, with the threshold of section 33. They are removed when this part is joined to the main notebook.

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

train_path=os.path.join("..","data","raw","train.csv")
clientes_path=os.path.join("..","data","raw","clientes.csv")

df_train= pd.read_csv(train_path,index_col="ID",sep=",")
df_clientes = pd.read_csv(clientes_path, sep=",")

target_col = "repeat_purchase_90d"
var_groups = {
    "Current transaction (financial)": ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Total Portes",
                                        "Total Desc_ Global", "Total Desc_ Linha", "Custo"],
    "Current transaction (operational)": ["N_ Detalhes", "invoice_count_current_purchase"],
    "Customer historical summary": ["customer_tenure_days", "prior_purchase_count", "prior_total_spend",
                                    "prior_mean_purchase_value", "prior_std_purchase_value",
                                    "prior_min_purchase_value", "prior_max_purchase_value"],
    "Customer cadence & gaps": ["days_since_previous_purchase", "prior_mean_gap_days", "prior_std_gap_days"],
    "Rolling windows (RFM)": ["purchase_count_30d", "spend_30d", "purchase_count_90d", "spend_90d",
                              "purchase_count_180d", "spend_180d", "purchase_count_365d", "spend_365d"],
}

# recover_values of 3.6.7
def recover_values(frame):
    """Fills the cells that can be computed exactly from other columns of the same row (3.6.1 and 3.6.7)."""
    out = frame.copy()
    count, tenure = out["prior_purchase_count"], out["customer_tenure_days"]
    out["prior_mean_purchase_value"] = out["prior_total_spend"] / count.replace(0, np.nan)

    days, gap = out["days_since_previous_purchase"].copy(), out["prior_mean_gap_days"].copy()
    fill = days.isna() & (count == 1)
    out.loc[fill, "days_since_previous_purchase"] = tenure[fill]
    fill = days.isna() & (count >= 2) & gap.notna()
    out.loc[fill, "days_since_previous_purchase"] = (tenure - gap * (count - 1))[fill]
    fill = gap.isna() & (count >= 2) & days.notna()
    out.loc[fill, "prior_mean_gap_days"] = ((tenure - days) / (count - 1))[fill]

    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == 0)
    out.loc[fill, "spend_365d"] = 0.0
    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == out["purchase_count_180d"])
    out.loc[fill, "spend_365d"] = out.loc[fill, "spend_180d"]
    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == count)
    out.loc[fill, "spend_365d"] = out.loc[fill, "prior_total_spend"]

    fill = out["prior_max_purchase_value"].isna() & (count == 1)
    out.loc[fill, "prior_max_purchase_value"] = out.loc[fill, "prior_total_spend"]
    fill = out["prior_max_purchase_value"].isna() & (count == 2)
    out.loc[fill, "prior_max_purchase_value"] = (out["prior_total_spend"] - out["prior_min_purchase_value"])[fill]
    return out

In [ ]:
import warnings
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, FunctionTransformer,
                                   StandardScaler, MinMaxScaler, RobustScaler)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score
from sklearn.exceptions import ConvergenceWarning

# without scaling the solver can stop at its limit of iterations. We look at this in section 18, so we hide the warning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
def signed_log(X):
    """log1p that keeps the sign. For values >= 0 it is the same as log1p, and it also works
    on the negative impossible values of the option "keep as is" (13.4)."""
    return np.sign(X) * np.log1p(np.abs(X))


class Winsorizer(BaseEstimator, TransformerMixin):
    """Clips each column to its 1% and 99% quantiles, learned on the train rows."""
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.low_ = np.nanquantile(X, 0.01, axis=0)
        self.high_ = np.nanquantile(X, 0.99, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)


def make_preprocessor(log_cols, plain_cols, cat_cols, flag_cols=(), num_fill="median", indicators=False,
                      cat_fill="Missing", outliers="log1p", encoder="onehot", min_frequency=None, scaler="standard"):
    """Layer B: the steps that learn something from the data (medians, levels, scales).
    Inside a Pipeline they are fitted only on the train rows of each split."""
    scalers = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}

    def numeric_steps(long_tail):
        if num_fill == "zero":
            steps = [("impute", SimpleImputer(strategy="constant", fill_value=0, add_indicator=indicators))]
        else:
            steps = [("impute", SimpleImputer(strategy=num_fill, add_indicator=indicators))]
        if long_tail and outliers == "log1p":
            steps.append(("log", FunctionTransformer(signed_log)))
        elif long_tail and outliers == "winsorize":
            steps.append(("winsorize", Winsorizer()))
        if scaler in scalers:
            steps.append(("scale", scalers[scaler]()))
        return Pipeline(steps)

    # categories: missing -> "Missing" (or the most frequent level), then encoding
    if cat_fill == "Missing":
        cat_steps = [("impute", SimpleImputer(strategy="constant", fill_value="Missing"))]
    else:
        cat_steps = [("impute", SimpleImputer(strategy="most_frequent"))]
    if encoder == "onehot" and min_frequency is None:
        cat_steps.append(("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)))
    elif encoder == "onehot":
        cat_steps.append(("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=min_frequency,
                                                  sparse_output=False)))
    else:
        cat_steps.append(("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)))
        if scaler in scalers:
            cat_steps.append(("scale", scalers[scaler]()))

    parts = [("log", numeric_steps(long_tail=True), list(log_cols)),       # money, counts and days (long tails)
             ("plain", numeric_steps(long_tail=False), list(plain_cols)),  # columns that can be negative, or ratios
             ("cat", Pipeline(cat_steps), list(cat_cols)),
             ("flags", "passthrough", list(flag_cols))]                    # 0/1 columns
    return ColumnTransformer([p for p in parts if len(p[2]) > 0])


def temporal_scores(frame, columns, recipe, row_filter=None, metric="ap"):
    """One score per fold: the instrument is fitted on the train rows and scored on the validation block."""
    used = columns["log_cols"] + columns["plain_cols"] + columns["cat_cols"] + columns["flag_cols"]
    scores = []
    for train_idx, val_idx in folds:
        train, val = frame.loc[train_idx], frame.loc[val_idx]
        if row_filter is not None:
            train = row_filter(train)   # rows are only removed from the train side, never from validation
        pipe = Pipeline([("prep", make_preprocessor(**columns, **recipe)),
                         ("model", LogisticRegression(max_iter=3000))])
        pipe.fit(train[used], train[target_col])
        prob = pipe.predict_proba(val[used])[:, 1]
        if metric == "ap":
            scores.append(average_precision_score(val[target_col], prob))
        elif metric == "f1":
            scores.append(f1_score(val[target_col], prob >= 0.5))
        else:
            scores.append((prob >= 0.5).mean())   # share of rows predicted as 1
    return np.array(scores)


def compare(results, reference):
    """Mean AP of each option and its difference to the reference, fold by fold (paired)."""
    rows = {}
    for name, scores in results.items():
        diff = scores - results[reference]
        sem = diff.std(ddof=1) / np.sqrt(len(diff))
        if name == reference:
            verdict = "reference"
        elif abs(diff.mean()) <= 2 * sem:
            verdict = "inside the noise"
        else:
            verdict = "better" if diff.mean() > 0 else "worse"
        rows[name] = {"AP": round(scores.mean(), 4), "difference": round(diff.mean(), 4), "SEM": round(sem, 4),
                      "better in": f"{(diff > 0).sum()}/{len(diff)}", "verdict": verdict}
    return pd.DataFrame(rows).T


def blank_impossible(frame):
    """Impossible values of 3.6 -> missing. It works row by row, so it learns nothing from the other rows."""
    out = frame.copy()
    count = out["prior_purchase_count"]
    # 3.6.1 to 3.6.3: 1e9, negative values, means bigger than the biggest purchase, spend_365d bigger than the prior spend
    mean = out["prior_mean_purchase_value"]
    out.loc[(mean < 0) | (mean >= 1e8) | (mean > out["prior_max_purchase_value"] + 0.01), "prior_mean_purchase_value"] = np.nan
    spend = out["spend_365d"]
    out.loc[(spend < 0) | (spend >= 1e8) | (spend > out["prior_total_spend"] + 0.01), "spend_365d"] = np.nan
    for col in ["days_since_previous_purchase", "prior_mean_gap_days"]:
        out.loc[out[col] < 0, col] = np.nan
    # 3.6.6: biggest purchase above its limit
    max_limit = out["prior_total_spend"] - (count - 1) * out["prior_min_purchase_value"]
    out.loc[(count >= 1) & (out["prior_max_purchase_value"] > max_limit + 0.01), "prior_max_purchase_value"] = np.nan
    return out


def add_features(frame):
    """New variables, built row by row from the columns of the same purchase."""
    out = frame.copy()
    months = out["customer_tenure_days"] / 30
    out["bought_in_90d"] = (out["purchase_count_90d"] > 0).astype(int)
    out["rhythm_ratio"] = out["days_since_previous_purchase"] / out["prior_mean_gap_days"].replace(0, np.nan)
    out["purchase_rate"] = out["prior_purchase_count"] / (months + 1)
    out["spend_per_month"] = out["prior_total_spend"] / (months + 1)
    out["avg_basket_365d"] = out["spend_365d"] / out["purchase_count_365d"].replace(0, np.nan)
    out["spend_acceleration"] = out["spend_30d"] / (out["spend_90d"] + 1)
    out["margin_ratio"] = (out["Total Liquido"] - out["Custo"]) / (out["Total Liquido"] + 1)
    out["discount_intensity"] = (out["Total Desc_ Linha"] + out["Total Desc_ Global"]) / (out["Total Bruto"] + 1)
    out["value_vs_habit"] = ((out["Total a Pagar"] - out["prior_mean_purchase_value"])
                             / out["prior_std_purchase_value"].replace(0, np.nan))
    return out


def prepare_data(frame, clientes):
    """Layer A: every step that works row by row. The same function will be used on test.csv."""
    out = frame.copy()
    out["purchase_date"] = pd.to_datetime(out["purchase_date"])
    out = blank_impossible(out)
    out = recover_values(out)          # 3.6.7
    out["is_first_purchase"] = (out["prior_purchase_count"] == 0).astype(int)
    out = out.join(clientes.set_index("customer_id")[["country_or_region", "commercial_region", "island",
                                                      "municipality", "postal_area"]],
                   on="customer_id", validate="many_to_one")
    out["incomplete_address"] = (out["municipality"].isna() & out["postal_area"].isna()).astype(int)
    out = add_features(out)
    return out

In [ ]:
# 12.1 and 12.2: rows without the empty ones, development rows and the 12 temporal splits
empty_rows = df_train.drop(columns=["customer_id", target_col]).isna().all(axis=1)
df_model = prepare_data(df_train.loc[~empty_rows], df_clientes)
is_dev = df_model["purchase_date"] < "2024-01-01"
dev = df_model.loc[is_dev]
block_starts = pd.date_range("2018-01-01", "2023-07-01", freq="6MS")
folds = []
for start in block_starts:
    end = start + pd.DateOffset(months=6)
    train_idx = dev.index[dev["purchase_date"] < start - pd.Timedelta(days=90)]
    val_idx = dev.index[(dev["purchase_date"] >= start) & (dev["purchase_date"] < end)]
    folds.append((train_idx, val_idx))

# 21.1: the candidates and the recipe that Part 2 hands to Part 3
final_columns = {
    "log_cols": [
        "Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Total Portes",
        "Total Desc_ Global", "Total Desc_ Linha", "Custo", "N_ Detalhes",
        "invoice_count_current_purchase", "customer_tenure_days", "prior_purchase_count",
        "prior_total_spend", "prior_mean_purchase_value", "prior_std_purchase_value",
        "prior_min_purchase_value", "prior_max_purchase_value", "days_since_previous_purchase",
        "prior_mean_gap_days", "prior_std_gap_days", "purchase_count_30d", "spend_30d",
        "purchase_count_90d", "spend_90d", "purchase_count_180d", "spend_180d", "purchase_count_365d",
        "spend_365d", "rhythm_ratio", "purchase_rate", "spend_per_month", "avg_basket_365d"],
    "plain_cols": [
        "random_noise", "spend_acceleration", "margin_ratio", "discount_intensity", "value_vs_habit"],
    "cat_cols": [
        "payment_type", "salesperson", "commercial_zone", "transport_method", "warehouse",
        "document_series", "country_or_region", "commercial_region", "island"],
    "flag_cols": [
        "is_first_purchase", "incomplete_address"],
}
final_recipe = {"num_fill": "median", "indicators": False, "cat_fill": "Missing", "outliers": "log1p", "encoder": "onehot", "min_frequency": 10, "scaler": "standard"}
feature_cols = final_columns["log_cols"] + final_columns["plain_cols"] + final_columns["cat_cols"] + final_columns["flag_cols"]
print(f"Rows: {len(df_model):,} | Development: {is_dev.sum():,} | Candidates: {len(feature_cols)}")

In [ ]:
# 28.3: the final set of Part 3
selected = ["Total Bruto", "prior_max_purchase_value", "prior_std_gap_days", "purchase_count_90d", "purchase_count_365d",
            "purchase_rate", "avg_basket_365d", "margin_ratio", "discount_intensity", "salesperson", "transport_method",
            "is_first_purchase"]
selected_columns = {group: [c for c in cols if c in selected] for group, cols in final_columns.items()}
print(f"Final set of Part 3: {len(selected)} variables")

In [ ]:
# The final model of Part 4 (sections 32 to 34): the average of the logistic regression (12 variables) and gradient
# boosting (48 candidates), both trained on 2015-2023 only, and the threshold of section 33.
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier

finalist_models = {"logistic regression": (LogisticRegression(C=0.1, max_iter=3000), selected_columns),
                   "gradient boosting": (HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=4,
                                                                        min_samples_leaf=80, max_iter=400,
                                                                        l2_regularization=1.0, random_state=44),
                                         final_columns)}


def fit_average(rows):
    """Fits the pipeline of each finalist on the given rows."""
    pipes = []
    for model, columns in finalist_models.values():
        pipe = Pipeline([("prep", make_preprocessor(**columns, **final_recipe)), ("model", clone(model))])
        pipes.append(pipe.fit(df_model.loc[rows, feature_cols], df_model.loc[rows, target_col]))
    return pipes


def predict_average(pipes, frame):
    """The mean of the probabilities of the two pipelines."""
    return np.mean([pipe.predict_proba(frame[feature_cols])[:, 1] for pipe in pipes], axis=0)


threshold = 0.37  # the best threshold of section 33 on our machines
dev_rows, holdout_rows = df_model.index[is_dev], df_model.index[~is_dev]
pipes = fit_average(dev_rows)
p_holdout = predict_average(pipes, df_model.loc[holdout_rows])
y_holdout = df_model.loc[holdout_rows, target_col].to_numpy()
print(f"Holdout: {len(holdout_rows)} purchases | repeat rate: {y_holdout.mean():.3f} | "
      f"AP: {average_precision_score(y_holdout, p_holdout):.3f} | F1 at {threshold}: {f1_score(y_holdout, p_holdout >= threshold):.3f}")

## **IV. Open-Ended: Turning the Model into a Contact Plan**

### **IV.0. Objective**
The company wants to use the model for more targeted commercial actions. In this section we turn its predictions into a contact plan and answer four questions: is the model worth using (H6), who to contact (H5), when to contact them (H2), and how to approach them, especially after a first purchase (H3 + H1). Each row of the data is a purchase, so all the shares below are shares of purchases, and the same customer can appear more than once.

### **IV.1. H6: Is the Model Worth Using?**

If the company can only contact part of its customers, the model should say where to start. On the holdout (2024 and 2025, 835 purchases) we sort the purchases by the risk of no repeat (1 − p) and count how many of the **lost purchases** (no new purchase of the customer within 90 days) are found as the contact list grows: this is the **gain curve**. We compare it with contacting at random and with a simple rule without a model, fixed before looking at the holdout: fewest purchases in the last 90 days first and, among ties, the most days since the previous purchase. The holdout is only used to measure, not to decide anything.

In [ ]:
# One row per holdout purchase, with the probability of a repeat (p) and the risk of no repeat (1 - p)
holdout = df_model.loc[holdout_rows].copy()
holdout["p"] = p_holdout
holdout["risk"] = 1 - holdout["p"]
holdout["lost"] = (holdout[target_col] == 0).astype(int)  # no other purchase of the customer within 90 days
n, n_lost = len(holdout), holdout["lost"].sum()

# Order of the model: highest risk first.
model_order = holdout.sort_values("risk", ascending=False, kind="stable").index
# Simple rule, fixed before looking at the holdout: fewest purchases in the last 90 days first and, among ties, most days
# since the previous purchase first. First purchases have no previous purchase, so they go first among their ties.
rule_order = (holdout.assign(days=holdout["days_since_previous_purchase"].fillna(np.inf))
              .sort_values(["purchase_count_90d", "days"], ascending=[True, False], kind="stable").index)


def found_share(order):
    """Share of the lost purchases found after contacting the first 1, 2, ..., n purchases of the order."""
    return holdout.loc[order, "lost"].cumsum().to_numpy() / n_lost


contacted = np.arange(1, n + 1) / n
gain = pd.DataFrame({"model": found_share(model_order), "simple rule": found_share(rule_order)}, index=contacted)
share_pred0 = (holdout["p"] < threshold).mean()  # purchases the model predicts as 0

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(gain.index * 100, gain["model"] * 100, color="#1779A8", label="model (highest risk first)")
ax.plot(gain.index * 100, gain["simple rule"] * 100, color="#C96A00", label="simple rule (purchases in the last 90 days)")
ax.plot([0, 100], [0, 100], color="grey", linestyle="--", linewidth=1, label="at random")
ax.axvline(share_pred0 * 100, color="black", linestyle=":", linewidth=1, label=f"purchases predicted as 0 ({share_pred0:.0%})")
ax.set(xlabel="% of the holdout purchases contacted", ylabel="% of the lost purchases found",
       title="Gain curve on the holdout (2024-2025)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

# The same curve at a few contact budgets
budgets = {"10%": 0.1, "20%": 0.2, "30%": 0.3, "50%": 0.5, f"predicted 0 ({share_pred0:.0%})": share_pred0}
rows = []
for name, b in budgets.items():
    k = int(round(b * n))
    rows.append({"contacted": name, "purchases": k,
                 "lost found, model": gain["model"].iloc[k - 1], "lost found, simple rule": gain["simple rule"].iloc[k - 1],
                 "lost found, at random": k / n, "lost among contacted, model": gain["model"].iloc[k - 1] * n_lost / k})
print(f"Lost purchases: {n_lost} of {n} ({n_lost / n:.0%})")
pd.DataFrame(rows).set_index("contacted").round(3)

About 40% of the holdout purchases are lost. Contacting the 20% with the highest risk finds about 38% of them, almost twice as many as at random (20%) and more than the simple rule (about 33%), and the model stays above the rule at almost every budget. The purchases the model predicts as 0 at the threshold of section 33 (about 23% of them) hold about 42% of the lost purchases, and about 3 in 4 of them are really lost. So the model is worth using, and this list is the natural starting point of the contact plan.

### **IV.2. H5: Who to Contact?**

Losing a large customer costs more than losing a small one, so we also rank the purchases by the **revenue at risk**: (1 − p) × the value of the next purchase. For this, p must be a realistic probability and not only a good order. We check it on the holdout: the purchases are split into five groups of the same size by p, and the mean p of each group is compared with its real repeat rate.

In [ ]:
# The holdout purchases in five groups of the same size by p: mean p against the real repeat rate
holdout["p group"] = pd.qcut(holdout["p"], 5, labels=["1 (lowest p)", "2", "3", "4", "5 (highest p)"])
calibration = holdout.groupby("p group", observed=True).agg(purchases=("p", "size"), mean_p=("p", "mean"),
                                                             repeat_rate=(target_col, "mean"))
print(f"All the holdout: mean p {holdout['p'].mean():.3f} | repeat rate {holdout[target_col].mean():.3f}")
calibration.round(3)

In every group the mean p is close to the real repeat rate (the largest gap is about 0.07, in the middle group), and over the whole holdout it is 0.59 against 0.61. So p can be used as a probability in the revenue at risk, with no correction.

For the value we use the mean value of the customer's purchases so far, the current one included: (`prior_total_spend` + `Total a Pagar`) / (`prior_purchase_count` + 1). It is one formula for every purchase, and for a first purchase it is simply the value of the purchase itself. We compare two lists of the same size, the purchases with the highest risk and those with the highest revenue at risk, and count the share of the lost purchases and of the **lost revenue** (the value of the lost purchases) that each one finds. The lost revenue is an estimate, the expected value of each lost purchase with the same formula, so it favours the list built on it and is a guide to the money at stake, not an exact amount.

In [ ]:
# Expected value of the next purchase: the mean value of the customer's purchases so far, the current one included.
# For a first purchase it is the value of the purchase itself.
holdout["value"] = (holdout["prior_total_spend"] + holdout["Total a Pagar"]) / (holdout["prior_purchase_count"] + 1)
holdout["revenue at risk"] = holdout["risk"] * holdout["value"]
lost_value = (holdout["value"] * holdout["lost"]).sum()  # the revenue that was really lost

orders = {"by risk": model_order,
          "by revenue at risk": holdout.sort_values("revenue at risk", ascending=False, kind="stable").index}
rows = []
for b in [0.1, 0.2, 0.3]:
    k = int(round(b * n))
    for name, order in orders.items():
        top = holdout.loc[order[:k]]
        rows.append({"contacted": f"{b:.0%}", "list": name, "lost purchases found": top["lost"].sum() / n_lost,
                     "lost revenue found": (top["value"] * top["lost"]).sum() / lost_value,
                     "customers": top["customer_id"].nunique()})
print(f"Lost revenue: {lost_value:,.0f} in {n_lost} purchases | the 10% most valuable lost purchases hold "
      f"{holdout.loc[holdout['lost'] == 1, 'value'].nlargest(int(round(0.1 * n_lost))).sum() / lost_value:.0%} of it")
pd.DataFrame(rows).set_index(["contacted", "list"]).round(3)

The lost revenue is concentrated: the 10% most valuable lost purchases hold about 30% of it. So, with the same budget, the list by revenue at risk finds about twice the lost revenue of the list by risk (about 50% against 26% when 20% of the purchases are contacted), but fewer lost purchases (about 26% against 38%). It also covers fewer customers (about 65 against 100), because large customers make many purchases.

In [ ]:
# Which purchases change list, for a budget of 20%
k = int(round(0.2 * n))
by_risk, by_revenue = set(orders["by risk"][:k]), set(orders["by revenue at risk"][:k])
groups = {"in both lists": by_risk & by_revenue, "only by risk": by_risk - by_revenue, "only by revenue at risk": by_revenue - by_risk}
pd.DataFrame({name: {"purchases": len(ids), "median value": holdout.loc[list(ids), "value"].median(),
                     "mean p": holdout.loc[list(ids), "p"].mean(), "repeat rate": holdout.loc[list(ids), target_col].mean(),
                     "first purchases": holdout.loc[list(ids), "is_first_purchase"].mean()}
              for name, ids in groups.items()}).T.round(3)

Only about a quarter of the contacted purchases are in both lists. The purchases that only the list by revenue at risk adds have a much higher value (median about 1,800 against 370), but also a higher chance of coming back (mean p about 0.51 against 0.26), and more than half of them did come back. Since the aim of the plan is to protect sales, we rank by **revenue at risk**: with the same number of contacts it protects about twice the revenue, even if some of the contacts go to customers who would have come back anyway.

In [ ]:
# The same comparison per customer. The revenue at risk of a customer is the sum over all their purchases in the
# holdout (2024 and 2025), so it is a total for the two years and not a list of calls for a single moment.
holdout["lost value"] = holdout["value"] * holdout["lost"]
customers = holdout.groupby("customer_id").agg(purchases=("p", "size"), mean_risk=("risk", "mean"),
                                               revenue_at_risk=("revenue at risk", "sum"),
                                               lost_revenue=("lost value", "sum"))
n_customers = len(customers)
rows = []
for b in [0.1, 0.2, 0.3]:
    k = int(round(b * n_customers))
    for name, column in {"by mean risk": "mean_risk", "by revenue at risk": "revenue_at_risk"}.items():
        top = customers.nlargest(k, column)
        rows.append({"customers contacted": f"{b:.0%} ({k})", "list": name,
                     "lost revenue found": top["lost_revenue"].sum() / customers["lost_revenue"].sum(),
                     "their share of the purchases": top["purchases"].sum() / n})
print(f"Customers in the holdout: {n_customers} | with at least one lost purchase: {(customers['lost_revenue'] > 0).sum()}"
      f" | purchases per customer: median {customers['purchases'].median():.0f}, max {customers['purchases'].max()}")
pd.DataFrame(rows).set_index(["customers contacted", "list"]).round(3)

Per customer, the revenue at risk is the sum over all the customer's purchases in the holdout, so it is a total for the two years (2024 and 2025), not a list of calls for a single moment. The 215 customers have a median of 3 purchases, but some have up to 36. The 20% of customers with the highest revenue at risk (43 customers) hold about 44% of the lost revenue, against about 13% for the 20% with the highest mean risk, and they make about 27% of the purchases. So a short list of large customers holds most of the money at risk, and these accounts are worth a regular follow-up, not a single call.

### **IV.3. H2: When to Contact?**

For each purchase of train.csv we take the date of the next purchase of the same customer (sorting by customer and date) and compute the days until it. This uses the future, so it is only a description of the data and **never enters the model**. The purchases of the last 90 days of train.csv are left out, because their next purchase can be after the end of the file. For the rest, a next purchase within 90 days agrees with the target in more than 99% of the rows, so the dates can be trusted.

In [ ]:
# Days until the next purchase of the same customer, from the dates of train.csv (sorted by customer and date).
# This uses the future, so it is only a description of the data and never enters the model.
seq = df_model.sort_values(["customer_id", "purchase_date"])[["customer_id", "purchase_date", "is_first_purchase", target_col]].copy()
seq["days_to_next"] = (seq.groupby("customer_id")["purchase_date"].shift(-1) - seq["purchase_date"]).dt.days
# The last 90 days of train.csv are left out: their next purchase can be after the end of the file
last_date = df_model["purchase_date"].max()
seq_end = seq["purchase_date"] > last_date - pd.Timedelta(days=90)
seq = seq.loc[~seq_end]
seq["back_90"] = seq["days_to_next"] <= 90
agreement = (seq["back_90"] == (seq[target_col] == 1)).mean()
print(f"Purchases: {len(seq):,} ({seq_end.sum()} of the last 90 days left out) | days to next purchase within 90 "
      f"agrees with the target in {agreement:.1%} of them")
back = seq.loc[seq["back_90"], "days_to_next"]
print(f"Repeat purchases: median {back.median():.0f} days | 25%: {back.quantile(0.25):.0f} | 75%: {back.quantile(0.75):.0f} "
      f"| 90%: {back.quantile(0.9):.0f}")


def still_comes_back(frame, t):
    """Of the purchases with no repeat by day t, the share with a repeat between day t + 1 and day 90."""
    waiting = frame[~(frame["days_to_next"] <= t)]
    return (waiting["days_to_next"] <= 90).mean()


days = np.arange(0, 90)
groups = {"later purchases": seq[seq["is_first_purchase"] == 0], "first purchases": seq[seq["is_first_purchase"] == 1]}
curves = pd.DataFrame({name: [still_comes_back(g, t) for t in days] for name, g in groups.items()}, index=days)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(back, bins=np.arange(0, 95, 5), color="#1779A8", edgecolor="white")
axes[0].set(xlabel="days until the next purchase", ylabel="repeat purchases", title="When the repeat purchases happen")
for (name, curve), color in zip(curves.items(), ["#1779A8", "#C96A00"]):
    axes[1].plot(curve.index, curve.values * 100, color=color, label=name)
axes[1].set(xlabel="day t after the purchase", ylabel="% that still comes back by day 90",
            title="Chance of a repeat by day 90, for purchases with no repeat by day t")
axes[1].legend()
plt.tight_layout()
plt.show()

steps = [0, 15, 30, 45, 60, 75]
pd.DataFrame({"repeats already happened": [(back <= t).mean() for t in steps],
              "still comes back, later purchases": curves.loc[steps, "later purchases"].to_numpy(),
              "still comes back, first purchases": curves.loc[steps, "first purchases"].to_numpy()},
             index=pd.Index(steps, name="day t")).round(3)

Repeat purchases happen early: half of them within about 23 days, 3 in 4 within about 49 days and 9 in 10 within 70 days. The chance of still coming back falls with every day without a purchase: for later purchases it goes from about 58% on the day of the purchase to 36% at day 30 and 19% at day 60, and first purchases start lower (about 34%) and are at 19% by day 30. A contact on the day of the purchase would also go to the 39% of repeats that happen in the first 15 days anyway, so we suggest contacting the purchases at risk that still have no new purchase **after about 15 days, and not later than day 30**.

In [ ]:
# Does the usual rhythm of the customer change the moment? The groups are fixed before looking at the results: thirds of
# prior_mean_gap_days. First purchases and purchases with only one earlier purchase have no usual gap, so they are left out.
rhythm = seq.join(df_model["prior_mean_gap_days"])
rhythm = rhythm[(rhythm["is_first_purchase"] == 0) & rhythm["prior_mean_gap_days"].notna()].copy()
rhythm["rhythm"] = pd.qcut(rhythm["prior_mean_gap_days"], 3, labels=["short", "medium", "long"])
rows = {}
for name, group in rhythm.groupby("rhythm", observed=True):
    back_days = group.loc[group["back_90"], "days_to_next"]
    rows[name] = {"purchases": len(group), "usual gap (days)": f"{group['prior_mean_gap_days'].min():.0f} to "
                                                               f"{group['prior_mean_gap_days'].max():.0f}",
                  "repeat within 90 days": round(group["back_90"].mean(), 3), "median day of the repeat": back_days.median(),
                  "repeats by day 15": round((back_days <= 15).mean(), 3),
                  **{f"still comes back, day {t}": round(still_comes_back(group, t), 3) for t in [15, 30, 45]}}
print(f"Left out: {len(seq) - len(rhythm)} purchases (first purchases or only one earlier purchase)")
pd.DataFrame(rows).T

The rhythm changes above all whether the customer comes back: about 83% of the purchases of the short group are followed by another within 90 days, against 60% for the medium group and 36% for the long one. It also changes the moment: in the short group (usual gap below about 51 days) half of the repeats happen within 15 days, while in the medium and long groups only about 30% do and the median repeat comes at about day 30. So for the short group a contact at about day 15 without a new purchase is right, while for the others no purchase by day 15 is still normal, and the contact can wait until about day 30.

### **IV.4. H3 + H1: How to Approach? Discounts and First Purchases**

First purchases come back much less often than later ones (H1: 0.33 against 0.58), so the first purchase is the moment where a new customer is won or lost. About 63% of the purchases have a discount (`Total Desc_ Linha` + `Total Desc_ Global` > 0), so we ask if purchases with a discount are followed by more repeat purchases (H3), for later and for first purchases. This is a description of all the rows of train.csv and does not use the model.

In [ ]:
# H1 and H3: repeat rate with and without a discount, for first and later purchases (all the rows of train.csv)
data = df_model.copy()
data["discount"] = ((data["Total Desc_ Linha"] + data["Total Desc_ Global"]) > 0).astype(int)
data["year"] = data["purchase_date"].dt.year
data["purchase"] = np.where(data["is_first_purchase"] == 1, "first purchase", "later purchase")
print(f"Repeat rate: first purchases {data.loc[data['is_first_purchase'] == 1, target_col].mean():.3f} | "
      f"later purchases {data.loc[data['is_first_purchase'] == 0, target_col].mean():.3f}")
print(f"Purchases with a discount: {data['discount'].mean():.0%} | by year: "
      + ", ".join(f"{y}: {v:.0%}" for y, v in data.groupby("year")["discount"].mean().items()))
data.groupby(["purchase", "discount"]).agg(purchases=(target_col, "size"), repeat_rate=(target_col, "mean"),
                                           purchases_last_365d=("purchase_count_365d", "mean"),
                                           purchases_last_90d=("purchase_count_90d", "mean"),
                                           median_gross_total=("Total Bruto", "median")).round(3)

The raw difference is large for later purchases (0.65 against 0.47 with and without a discount) and smaller for first purchases (0.36 against 0.29). But it is misleading: later purchases with a discount come from much more active customers (about 9 purchases in the last 365 days against 4) and are larger, and the share of purchases with a discount grows from about 50% in 2015 to 83% in 2025. So the difference mixes the discount with the type of customer and the year.

To compare like with like, we **match** each purchase with a discount to the most similar purchase without a discount from the same year, with `NearestNeighbors(n_neighbors=1)`, the same neighbour search as KNN. The variables go through log1p and are standardized, so the distance is in standard deviations, and a purchase without a discount can be the partner of more than one purchase. A pair is kept only if the typical difference per variable (the distance divided by the square root of the number of variables) is at most 0.2 standard deviations: this is the **caliper**, and the purchases with no partner that close are left out. We check the pairs with the **SMD** (the difference of the means divided by the standard deviation; below 0.1 the two groups are alike), and we get a 95% interval with a **bootstrap by customer**: 1,000 times we draw customers at random with replacement, keep all their pairs and recompute the difference, because the purchases of the same customer are not independent.

In [ ]:
from sklearn.neighbors import NearestNeighbors


def match(frame, variables, stratum, caliper=0.2):
    """For each purchase with a discount, the closest purchase without a discount in the same stratum (year or period).
    The variables go through log1p and are standardized, so the distance is measured in standard deviations.
    A purchase without a discount can be used more than once."""
    X = pd.DataFrame(StandardScaler().fit_transform(np.log1p(frame[variables])), index=frame.index, columns=variables)
    pairs = []
    for s in sorted(frame[stratum].unique()):
        treated = frame.index[(frame[stratum] == s) & (frame["discount"] == 1)]
        controls = frame.index[(frame[stratum] == s) & (frame["discount"] == 0)]
        if len(treated) == 0 or len(controls) == 0:
            continue
        distance, position = NearestNeighbors(n_neighbors=1).fit(X.loc[controls]).kneighbors(X.loc[treated])
        pairs.append(pd.DataFrame({"treated": treated, "control": controls[position[:, 0]], "distance": distance[:, 0]}))
    pairs = pd.concat(pairs, ignore_index=True)
    # Caliper: the distance divided by the square root of the number of variables is the typical difference per
    # variable. Pairs where it is above 0.2 standard deviations are dropped.
    pairs["per_variable"] = pairs["distance"] / np.sqrt(len(variables))
    return pairs[pairs["per_variable"] <= caliper].reset_index(drop=True), X


def smd(X, treated, control):
    """Standardized mean difference: difference of the means divided by the mean standard deviation of the two groups."""
    a, b = X.loc[treated], X.loc[control]
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)


def bootstrap_by_customer(frame, pairs, n_boot=1000):
    """95% interval of the difference in repeat rate. Each round draws customers with replacement and keeps all
    their pairs, because the purchases of the same customer are not independent."""
    diff = frame.loc[pairs["treated"], target_col].to_numpy() - frame.loc[pairs["control"], target_col].to_numpy()
    by_customer = pd.Series(diff).groupby(frame.loc[pairs["treated"], "customer_id"].to_numpy())
    sums, counts = by_customer.sum().to_numpy(), by_customer.size().to_numpy()
    draws = np.random.default_rng(44).integers(0, len(sums), size=(n_boot, len(sums)))
    return np.percentile(sums[draws].sum(axis=1) / counts[draws].sum(axis=1), [2.5, 97.5])


def matching_report(frame, variables, stratum):
    """Runs the matching and prints the pairs, the balance table and the result with its interval."""
    missing = frame[variables].isna().any(axis=1)
    if missing.any():
        print(f"Left out because a matching variable is missing: {missing.sum()} purchases")
    frame = frame[~missing]
    pairs, X = match(frame, variables, stratum)
    n_treated = frame["discount"].sum()
    treated_all, control_all = frame.index[frame["discount"] == 1], frame.index[frame["discount"] == 0]
    y = frame[target_col]
    rate_t, rate_c = y.loc[pairs["treated"]].mean(), y.loc[pairs["control"]].mean()
    low, high = bootstrap_by_customer(frame, pairs)
    print(f"With a discount: {n_treated} | matched: {len(pairs)} ({len(pairs) / n_treated:.0%}), "
          f"left out by the caliper: {n_treated - len(pairs)} | different purchases without a discount used: {pairs['control'].nunique()}"
          f" of {len(control_all)}")
    print(f"Before matching: {y.loc[treated_all].mean():.3f} with a discount vs {y.loc[control_all].mean():.3f} without "
          f"({y.loc[treated_all].mean() - y.loc[control_all].mean():+.3f})")
    print(f"After matching:  {rate_t:.3f} with a discount vs {rate_c:.3f} in the pairs ({rate_t - rate_c:+.3f}), "
          f"95% interval {low:+.3f} to {high:+.3f}")
    return pd.DataFrame({"SMD before": smd(X, treated_all, control_all),
                         "SMD after": smd(X, pairs["treated"], pairs["control"])}).round(3)


later = data[data["is_first_purchase"] == 0]
first = data[data["is_first_purchase"] == 1].copy()
first["period"] = pd.cut(first["year"], [2014, 2018, 2021, 2025], labels=["2015-2018", "2019-2021", "2022-2025"])
later_vars = ["purchase_count_90d", "purchase_count_365d", "purchase_rate", "days_since_previous_purchase", "Total Bruto"]
first_vars = ["Total Bruto", "N_ Detalhes"]

In [ ]:
# Later purchases: matched on the history of the customer and the size of the purchase, within the same year
matching_report(later, later_vars, "year")

Later purchases are matched on the recent activity of the customer (`purchase_count_90d`, `purchase_count_365d`, `purchase_rate` and `days_since_previous_purchase`, used instead of `prior_mean_gap_days` because that one is missing when the customer has only one earlier purchase) and on the size of the purchase (`Total Bruto`). About 48% of the purchases with a discount find a close partner, and after matching every SMD is below 0.02, against up to 0.53 before. The difference falls from +0.18 to about +0.04 (95% interval about +0.02 to +0.07): most of the raw gap comes from who receives the discount, and only a small gain is left.

In [ ]:
# The same matching with other calipers, to see how much the result depends on this choice
known = later.dropna(subset=later_vars)
rows = []
for caliper in [0.1, 0.2, 0.3, 0.5, 1.0]:
    pairs, X = match(known, later_vars, "year", caliper)
    y = known[target_col]
    rows.append({"caliper": caliper, "matched": len(pairs) / known["discount"].sum(),
                 "largest |SMD| after": smd(X, pairs["treated"], pairs["control"]).abs().max(),
                 "difference": y.loc[pairs["treated"]].mean() - y.loc[pairs["control"]].mean()})
pd.DataFrame(rows).set_index("caliper").round(3)

The caliper decides how close the pairs must be. A stricter caliper matches only about 15% of the purchases with a discount and a looser one up to about 94%, but the difference always stays between about +0.04 and +0.08 and every SMD stays below 0.1. So the small gain does not depend on this choice.

In [ ]:
# First purchases: no history, so they are matched on the purchase itself. Within the same year only a few pairs are
# left, so they are matched within three periods.
print(f"First purchases without a discount per year: {first.loc[first['discount'] == 0, 'year'].value_counts().sort_index().to_dict()}")
year_pairs = match(first, first_vars, "year")[0]
print(f"Within the same year: {len(year_pairs)} pairs, difference "
      f"{first.loc[year_pairs['treated'], target_col].mean() - first.loc[year_pairs['control'], target_col].mean():+.3f}")
matching_report(first, first_vars, "period")

First purchases have no history, so they are matched on the purchase itself (`Total Bruto` and `N_ Detalhes`). Within the same year only 173 pairs are left, because from 2022 there are at most 6 first purchases without a discount per year, so we match within three periods (2015-2018, 2019-2021 and 2022-2025), which gives 239 pairs (67%). The difference rises from +0.07 to about +0.18 (95% interval about +0.10 to +0.26), and within the same year it is +0.13. The gain is clearly larger than for later purchases, but it rests on few purchases: only 127 different purchases without a discount are used, many of them more than once, so the real uncertainty is larger than the interval.

In [ ]:
# The first purchases of 2015 may be older customers loaded into the system without their history (section 9),
# so we repeat the matching without them.
print(f"Repeat rate of first purchases: 2015 {first.loc[first['year'] == 2015, target_col].mean():.3f} | "
      f"after 2015 {first.loc[first['year'] > 2015, target_col].mean():.3f}")
first_no2015 = first[first["year"] > 2015].copy()
first_no2015["period"] = pd.cut(first_no2015["year"], [2015, 2018, 2021, 2025], labels=["2016-2018", "2019-2021", "2022-2025"])
matching_report(first_no2015, first_vars, "period")

The first purchases of 2015 come back much more often than later first purchases (0.45 against 0.29, section 9), probably because many are older customers loaded into the system without their history. Without them the gain is smaller but still clear: about +0.13, with a 95% interval from about +0.04 to +0.22.

Both results are an **association, not a cause**: the pairs are alike on the variables we used, but they can differ in things we do not see, for example why a salesperson decided to give the discount. The signal is strongest where it matters most, the first purchase, where a discount comes with about 13 to 18 more repeats per 100 purchases, against about 4 for later purchases. Before turning this into a policy, we recommend an **A/B test**: give a discount at random to half of the new customers and compare their repeat rate after 90 days.

### **IV.5. Key Takeaways**

- **Is the model worth using? (H6)** Yes. Contacting the 20% of the purchases with the highest risk finds about 38% of the lost purchases, against 20% at random and about 33% with a simple rule on recent purchases.
- **Who to contact? (H5)** Rank by revenue at risk, (1 − p) × the mean value of the customer's purchases. The probabilities are realistic, and with the same budget this list finds about twice the lost revenue (about 50% against 26% at 20% of contacts). Per customer, over the two years of the holdout, the 20% of customers with the highest revenue at risk hold about 44% of the lost revenue.
- **When to contact? (H2)** Early, and depending on the rhythm of the customer. Half of the repeats happen within about 23 days, and the chance of a repeat falls with every day without a purchase. Customers who usually buy every 51 days or less should be contacted at about day 15 without a new purchase, the others at about day 30.
- **How to approach? (H3 + H1)** First purchases come back much less (0.33 against 0.58). A discount on a first purchase comes with about 13 to 18 more repeats per 100 purchases than similar first purchases without one, against about 4 for later purchases, so discounts are more useful for new customers than for the most active ones, who receive most of them today.
- **Next step.** These are associations, not causes. An A/B test with random discounts on first purchases would show if the effect is real before the discount policy is changed.